In [1]:
pip install numpy pandas scipy scikit-learn matplotlib seaborn openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ================================================================
# EMPIRICAL ANALYSIS
# An Overlap-Adaptive Doubly Robust Estimator for Transporting
# Clinical Treatment Effects Under Covariate Shift and
# Outcome Missingness
#
# Dataset: Negi_dataset.csv
#
# Components:
# 1. Experimental NSW benchmark
# 2. PSID-1 transported treatment effect
# 3. PSID-2 transported treatment effect
# 4. OR, IPW and DR estimators
# 5. Fixed-score and Estimated-overlap bootstrap
# 6. Linear and Quadratic outcome-model sensitivity
# 7. Bootstrap confidence intervals
# 8. ESS, overlap, weight and covariate-balance diagnostics
# 9. Tables, CSV files, Excel workbook and figures
# ================================================================


# ================================================================
# 0. IMPORT PACKAGES
# ================================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, LinearRegression

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")


# ================================================================
# 1. USER SETTINGS
# ================================================================

DATA_PATH = r"E:\Downloaded Files\Negi_dataset.csv"

OUTPUT_DIR = r"D:\empirical analysis"

# Adaptive overlap tuning parameters
A_VALUES = [0.0, 0.005, 0.01, 0.05, 0.10]

# Number of bootstrap replications
B_BOOT = 1000

# Random seed
SEED = 2026

# Probability clipping
EPS = 1e-4

# NSW randomized treatment allocation probability.
# The NSW source sample is treated as a 1:1 randomized experiment.
P_RCT = 0.50

# Outcome-model specifications
OUTCOME_SPECS = ["LINEAR", "QUADRATIC"]

# Covariates used in all analyses
COVARIATES = [
    "age",
    "educ",
    "nodegree",
    "married",
    "black",
    "hisp",
    "nchildren75",
    "re74",
    "re75"
]

OUTCOME = "re79"
TREATMENT = "treated"
SOURCE_INDICATOR = "sample_lalonde"


# ================================================================
# 2. CREATE OUTPUT FOLDER
# ================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

FIG_DIR = os.path.join(OUTPUT_DIR, "Figures")
TABLE_DIR = os.path.join(OUTPUT_DIR, "Tables")
BOOT_DIR = os.path.join(OUTPUT_DIR, "Bootstrap")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(BOOT_DIR, exist_ok=True)

rng = np.random.default_rng(SEED)


# ================================================================
# 3. LOAD DATA
# ================================================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please change DATA_PATH to the correct CSV location."
    )

df = pd.read_csv(DATA_PATH)

required_columns = (
    COVARIATES
    + [
        OUTCOME,
        TREATMENT,
        SOURCE_INDICATOR,
        "psid1",
        "psid2"
    ]
)

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

# Preserve original row identifier for fixed-score bootstrap
df = df.copy()
df["_rid"] = np.arange(len(df))

# Source population: NSW
df["Z"] = (df[SOURCE_INDICATOR] == 1).astype(int)

# Treatment indicator is defined only for source observations
df["W"] = pd.to_numeric(df[TREATMENT], errors="coerce")

# Outcome observation indicator
df["S"] = 0
source_mask = df["Z"] == 1

df.loc[source_mask, "S"] = (
    df.loc[source_mask, OUTCOME].notna().astype(int)
)

# Convert covariates to numeric
for col in COVARIATES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[OUTCOME] = pd.to_numeric(df[OUTCOME], errors="coerce")


# ================================================================
# 4. DEFINE SOURCE AND TARGET POPULATIONS
# ================================================================

source = df[df["Z"] == 1].copy()

target_psid1 = df[
    (df["Z"] == 0) &
    (df["psid1"] == 1)
].copy()

target_psid2 = df[
    (df["Z"] == 0) &
    (df["psid2"].fillna(0) == 1)
].copy()

# Treatment values in the source population
source = source[source["W"].isin([0, 1])].copy()

# Ensure outcome is not used from target population
target_psid1[OUTCOME] = np.nan
target_psid2[OUTCOME] = np.nan

TARGETS = {
    "PSID-1": target_psid1,
    "PSID-2": target_psid2
}


# ================================================================
# 5. SAMPLE FLOW TABLE
# ================================================================

sample_flow = pd.DataFrame([
    {
        "Population": "Full dataset",
        "Total N": len(df),
        "Observed re79": df[OUTCOME].notna().sum()
    },
    {
        "Population": "NSW source",
        "Total N": len(source),
        "Observed re79": source[OUTCOME].notna().sum()
    },
    {
        "Population": "NSW treated",
        "Total N": (source["W"] == 1).sum(),
        "Observed re79": source.loc[
            source["W"] == 1, OUTCOME
        ].notna().sum()
    },
    {
        "Population": "NSW control",
        "Total N": (source["W"] == 0).sum(),
        "Observed re79": source.loc[
            source["W"] == 0, OUTCOME
        ].notna().sum()
    },
    {
        "Population": "PSID-1",
        "Total N": len(target_psid1),
        "Observed re79": target_psid1[OUTCOME].notna().sum()
    },
    {
        "Population": "PSID-2",
        "Total N": len(target_psid2),
        "Observed re79": target_psid2[OUTCOME].notna().sum()
    }
])

sample_flow.to_csv(
    os.path.join(TABLE_DIR, "Table_1_Sample_Flow.csv"),
    index=False
)


# ================================================================
# 6. BASIC SAMPLE DESCRIPTIVES
# ================================================================

descriptive_rows = []

for pop_name, pop_data in [
    ("NSW", source),
    ("PSID-1", target_psid1),
    ("PSID-2", target_psid2)
]:

    for variable in COVARIATES:

        descriptive_rows.append({
            "Population": pop_name,
            "Variable": variable,
            "N": pop_data[variable].notna().sum(),
            "Mean": pop_data[variable].mean(),
            "SD": pop_data[variable].std(),
            "Minimum": pop_data[variable].min(),
            "Maximum": pop_data[variable].max()
        })

descriptive_table = pd.DataFrame(descriptive_rows)

descriptive_table.to_csv(
    os.path.join(TABLE_DIR, "Table_2_Descriptive_Statistics.csv"),
    index=False
)


# ================================================================
# 7. HELPER FUNCTIONS
# ================================================================

def clip_probability(x):
    return np.clip(np.asarray(x, dtype=float), EPS, 1 - EPS)


def fit_binary_model(X, y):

    y = np.asarray(y, dtype=int)

    # Handle constant outcome in rare bootstrap samples
    if len(np.unique(y)) < 2:
        return ("constant", float(np.mean(y)))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(
            C=1e4,
            solver="lbfgs",
            max_iter=5000
        ))
    ])

    model.fit(X, y)

    return ("model", model)


def predict_binary_model(fitted, X):

    model_type, model = fitted

    if model_type == "constant":
        pred = np.repeat(model, len(X))
    else:
        pred = model.predict_proba(X)[:, 1]

    return clip_probability(pred)


def fit_outcome_model(X, y, specification="LINEAR"):

    if specification == "LINEAR":

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("regression", LinearRegression())
        ])

    elif specification == "QUADRATIC":

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("polynomial", PolynomialFeatures(
                degree=2,
                include_bias=False
            )),
            ("regression", LinearRegression())
        ])

    else:
        raise ValueError("Unknown outcome-model specification.")

    model.fit(X, np.asarray(y, dtype=float))

    return model


# ================================================================
# 8. FIT TRANSPORT NUISANCE MODELS
# ================================================================

def fit_transport_models(data, outcome_spec="LINEAR"):

    X = data[COVARIATES]

    source_mask = data["Z"].to_numpy() == 1

    source_data = data.loc[source_mask].copy()

    # ------------------------------------------------------------
    # 8.1 Membership model e(X) = P(Z=1 | X)
    # ------------------------------------------------------------

    membership_model = fit_binary_model(
        X,
        data["Z"].to_numpy()
    )

    e_hat = predict_binary_model(
        membership_model,
        X
    )

    # ------------------------------------------------------------
    # 8.2 Treatment probability
    # ------------------------------------------------------------
    # NSW is a randomized 1:1 source experiment.
    # Therefore, the design-known treatment probability is used.
    # This avoids treating randomized assignment as observational
    # confounding.

    p1_hat = np.repeat(P_RCT, len(data))
    p0_hat = 1.0 - p1_hat

    # ------------------------------------------------------------
    # 8.3 Treatment-specific outcome observation models
    # ------------------------------------------------------------

    response_models = {}

    for g in [0, 1]:

        arm_mask = (
            (source_data["W"].to_numpy() == g)
        )

        arm_data = source_data.loc[arm_mask]

        response_models[g] = fit_binary_model(
            arm_data[COVARIATES],
            arm_data["S"].to_numpy()
        )

    r1_hat = predict_binary_model(
        response_models[1],
        X
    )

    r0_hat = predict_binary_model(
        response_models[0],
        X
    )

    # ------------------------------------------------------------
    # 8.4 Treatment-specific outcome regression models
    # ------------------------------------------------------------

    outcome_models = {}

    for g in [0, 1]:

        arm_observed = source_data[
            (source_data["W"] == g) &
            (source_data["S"] == 1)
        ].copy()

        if len(arm_observed) < 5:
            raise ValueError(
                f"Too few observed outcomes in treatment arm {g}."
            )

        outcome_models[g] = fit_outcome_model(
            arm_observed[COVARIATES],
            arm_observed[OUTCOME],
            specification=outcome_spec
        )

    m1_hat = outcome_models[1].predict(X)
    m0_hat = outcome_models[0].predict(X)

    # ------------------------------------------------------------
    # 8.5 Joint treatment-specific support
    # ------------------------------------------------------------

    tau1 = e_hat * r1_hat * p1_hat
    tau0 = e_hat * r0_hat * p0_hat

    tau1 = np.maximum(tau1, EPS)
    tau0 = np.maximum(tau0, EPS)

    tau = (tau1 * tau0) / (tau1 + tau0)

    return {
        "e": e_hat,
        "p1": p1_hat,
        "p0": p0_hat,
        "r1": r1_hat,
        "r0": r0_hat,
        "m1": m1_hat,
        "m0": m0_hat,
        "tau1": tau1,
        "tau0": tau0,
        "tau": tau
    }


# ================================================================
# 9. ADAPTIVE OVERLAP FUNCTION
# ================================================================

def overlap_function(tau, a):

    tau = np.asarray(tau, dtype=float)

    if a == 0:
        return np.ones(len(tau))

    return tau / (tau + a)


# ================================================================
# 10. TRANSPORTED OR, IPW AND DR ESTIMATORS
# ================================================================

def calculate_transport_estimators(
    data,
    fitted,
    a,
    phi_fixed=None
):

    Z = data["Z"].to_numpy()
    W = data["W"].to_numpy()
    S = data["S"].to_numpy()
    Y = data[OUTCOME].to_numpy(dtype=float)

    e = fitted["e"]
    p1 = fitted["p1"]
    p0 = fitted["p0"]
    r1 = fitted["r1"]
    r0 = fitted["r0"]
    m1 = fitted["m1"]
    m0 = fitted["m0"]

    target_mask = Z == 0
    source_mask = Z == 1

    # Estimated-overlap score
    phi_estimated = overlap_function(
        fitted["tau"], a
    )

    # Fixed-score empirical overlap approximation
    if phi_fixed is None:
        phi_fixed = phi_estimated.copy()

    results = []

    for track, phi in [
        ("FIXED_SCORE", phi_fixed),
        ("ESTIMATED_OVERLAP", phi_estimated)
    ]:

        target_phi = phi[target_mask]
        target_denominator = np.sum(target_phi)

        if target_denominator <= 0:
            continue

        for g in [1, 0]:

            if g == 1:
                pg = p1
                rg = r1
                mg = m1
            else:
                pg = p0
                rg = r0
                mg = m0

            # Triple weight without the Z*S*I(W=g) indicator
            omega = ((1 - e) / e) / (pg * rg)

            arm_mask = (
                source_mask &
                (W == g) &
                (S == 1) &
                np.isfinite(Y)
            )

            source_weights = (
                phi[arm_mask] *
                omega[arm_mask]
            )

            source_y = Y[arm_mask]

            if (
                len(source_y) == 0 or
                np.sum(source_weights) <= 0
            ):
                raise ValueError(
                    f"No usable source observations in arm {g}."
                )

            # ----------------------------------------------------
            # OR: outcome-regression plug-in
            # ----------------------------------------------------

            target_mg = mg[target_mask]

            mean_or = (
                np.sum(target_phi * target_mg) /
                target_denominator
            )

            # ----------------------------------------------------
            # IPW: Hájek-normalized estimator
            # ----------------------------------------------------

            mean_ipw = (
                np.sum(source_weights * source_y) /
                np.sum(source_weights)
            )

            # ----------------------------------------------------
            # DR: augmented transported mean
            # ----------------------------------------------------

            source_mg = mg[arm_mask]

            residual_component = np.sum(
                source_weights *
                (source_y - source_mg)
            )

            mean_dr = (
                np.sum(target_phi * target_mg) +
                residual_component
            ) / target_denominator

            # ----------------------------------------------------
            # Effective sample size
            # ----------------------------------------------------

            ess = (
                np.sum(source_weights) ** 2 /
                np.sum(source_weights ** 2)
            )

            results.append({
                "Track": track,
                "Treatment_Arm": g,
                "OR_Mean": mean_or,
                "IPW_Mean": mean_ipw,
                "DR_Mean": mean_dr,
                "ESS": ess,
                "Mean_Phi": np.mean(target_phi),
                "Mean_Tau": np.mean(
                    fitted["tau"][target_mask]
                ),
                "Max_Weight": np.max(source_weights)
            })

    # Convert arm-specific means into treatment effects
    result_df = pd.DataFrame(results)

    final_rows = []

    for track in result_df["Track"].unique():

        part = result_df[
            result_df["Track"] == track
        ]

        treated = part[
            part["Treatment_Arm"] == 1
        ].iloc[0]

        control = part[
            part["Treatment_Arm"] == 0
        ].iloc[0]

        final_rows.append({
            "Track": track,
            "OR": treated["OR_Mean"] - control["OR_Mean"],
            "IPW": treated["IPW_Mean"] - control["IPW_Mean"],
            "DR": treated["DR_Mean"] - control["DR_Mean"],
            "ESS1": treated["ESS"],
            "ESS0": control["ESS"],
            "Mean_Phi": treated["Mean_Phi"],
            "Mean_Tau": treated["Mean_Tau"],
            "Max_Weight1": treated["Max_Weight"],
            "Max_Weight0": control["Max_Weight"]
        })

    return pd.DataFrame(final_rows)


# ================================================================
# 11. FIT ORIGINAL-SAMPLE TRANSPORT MODELS
# ================================================================

def prepare_transport_data(target_data):

    combined = pd.concat(
        [source, target_data],
        axis=0,
        ignore_index=True
    )

    combined = combined.copy()

    # Source indicator already exists
    combined["Z"] = combined["Z"].astype(int)

    # Target outcomes are never used
    combined.loc[
        combined["Z"] == 0, OUTCOME
    ] = np.nan

    combined.loc[
        combined["Z"] == 0, "S"
    ] = 0

    return combined


# ================================================================
# 12. TRANSPORT BOOTSTRAP
# ================================================================

def run_transport_analysis(
    target_name,
    target_data,
    outcome_spec="LINEAR"
):

    print("\n" + "=" * 70)
    print(f"TRANSPORT ANALYSIS: {target_name}")
    print(f"Outcome specification: {outcome_spec}")
    print("=" * 70)

    data = prepare_transport_data(target_data)

    # Original-sample nuisance models
    fitted_original = fit_transport_models(
        data,
        outcome_spec
    )

    # Original-sample fixed overlap score map
    original_tau_map = dict(
        zip(data["_rid"], fitted_original["tau"])
    )

    point_rows = []
    diagnostic_rows = []
    balance_rows = []

    for a in A_VALUES:

        phi_original = overlap_function(
            fitted_original["tau"], a
        )

        # Both tracks have the same point estimate on original data.
        point = calculate_transport_estimators(
            data,
            fitted_original,
            a,
            phi_fixed=phi_original
        )

        for _, row in point.iterrows():

            point_rows.append({
                "Target": target_name,
                "Outcome_Spec": outcome_spec,
                "a": a,
                "Track": row["Track"],
                "OR": row["OR"],
                "IPW": row["IPW"],
                "DR": row["DR"]
            })

            diagnostic_rows.append({
                "Target": target_name,
                "Outcome_Spec": outcome_spec,
                "a": a,
                "Track": row["Track"],
                "Mean_Phi": row["Mean_Phi"],
                "Mean_Tau": row["Mean_Tau"],
                "ESS1": row["ESS1"],
                "ESS0": row["ESS0"],
                "Max_Weight1": row["Max_Weight1"],
                "Max_Weight0": row["Max_Weight0"]
            })

    # ------------------------------------------------------------
    # Bootstrap
    # ------------------------------------------------------------

    bootstrap_rows = []
    successful_bootstraps = 0

    for b in range(B_BOOT):

        boot = data.sample(
            n=len(data),
            replace=True,
            random_state=int(rng.integers(1, 2**31 - 1))
        ).reset_index(drop=True)

        try:

            fitted_boot = fit_transport_models(
                boot,
                outcome_spec
            )

            # Fixed-score bootstrap:
            # Freeze the original empirical overlap score for each
            # original row; nuisance models are refitted.
            fixed_tau = np.array([
                original_tau_map[rid]
                for rid in boot["_rid"]
            ])

            for a in A_VALUES:

                phi_fixed = overlap_function(
                    fixed_tau, a
                )

                boot_estimates = calculate_transport_estimators(
                    boot,
                    fitted_boot,
                    a,
                    phi_fixed=phi_fixed
                )

                for _, row in boot_estimates.iterrows():

                    for estimator in ["OR", "IPW", "DR"]:

                        bootstrap_rows.append({
                            "Target": target_name,
                            "Outcome_Spec": outcome_spec,
                            "Bootstrap": b + 1,
                            "a": a,
                            "Track": row["Track"],
                            "Estimator": estimator,
                            "Estimate": row[estimator]
                        })

            successful_bootstraps += 1

        except Exception as error:

            print(
                f"Bootstrap {b + 1} failed for "
                f"{target_name}, {outcome_spec}: {error}"
            )

    print(
        f"Successful bootstrap replications: "
        f"{successful_bootstraps}/{B_BOOT}"
    )

    bootstrap_df = pd.DataFrame(bootstrap_rows)

    # Save raw bootstrap estimates
    bootstrap_file = (
        f"Bootstrap_{target_name.replace('-', '')}_"
        f"{outcome_spec}.csv"
    )

    bootstrap_df.to_csv(
        os.path.join(BOOT_DIR, bootstrap_file),
        index=False
    )

    # ------------------------------------------------------------
    # Confidence intervals and standard errors
    # ------------------------------------------------------------

    summary_rows = []

    for _, point in pd.DataFrame(point_rows).iterrows():

        for estimator in ["OR", "IPW", "DR"]:

            boot_part = bootstrap_df[
                (bootstrap_df["Target"] == point["Target"]) &
                (bootstrap_df["Outcome_Spec"] == point["Outcome_Spec"]) &
                (bootstrap_df["a"] == point["a"]) &
                (bootstrap_df["Track"] == point["Track"]) &
                (bootstrap_df["Estimator"] == estimator)
            ]["Estimate"].dropna()

            if len(boot_part) > 1:

                lower = boot_part.quantile(0.025)
                upper = boot_part.quantile(0.975)
                se = boot_part.std(ddof=1)

            else:

                lower = np.nan
                upper = np.nan
                se = np.nan

            summary_rows.append({
                "Target": point["Target"],
                "Outcome_Spec": point["Outcome_Spec"],
                "a": point["a"],
                "Track": point["Track"],
                "Estimator": estimator,
                "Estimate": point[estimator],
                "Bootstrap_SE": se,
                "CI_Lower": lower,
                "CI_Upper": upper,
                "Successful_Bootstrap": len(boot_part)
            })

    summary_df = pd.DataFrame(summary_rows)

    diagnostics_df = pd.DataFrame(diagnostic_rows)

    return (
        summary_df,
        diagnostics_df,
        bootstrap_df,
        data,
        fitted_original
    )


# ================================================================
# 13. EXPERIMENTAL NSW BENCHMARK
# ================================================================

def fit_rct_models(data, outcome_spec="LINEAR"):

    X = data[COVARIATES]

    response_models = {}
    outcome_models = {}

    for g in [0, 1]:

        arm = data[data["W"] == g].copy()

        response_models[g] = fit_binary_model(
            arm[COVARIATES],
            arm["S"].to_numpy()
        )

        observed_arm = arm[
            (arm["S"] == 1) &
            (arm[OUTCOME].notna())
        ]

        outcome_models[g] = fit_outcome_model(
            observed_arm[COVARIATES],
            observed_arm[OUTCOME],
            outcome_spec
        )

    r1 = predict_binary_model(
        response_models[1], X
    )

    r0 = predict_binary_model(
        response_models[0], X
    )

    m1 = outcome_models[1].predict(X)
    m0 = outcome_models[0].predict(X)

    return {
        "r1": r1,
        "r0": r0,
        "m1": m1,
        "m0": m0
    }


def calculate_rct_estimators(data, fitted):

    W = data["W"].to_numpy()
    S = data["S"].to_numpy()
    Y = data[OUTCOME].to_numpy(dtype=float)

    r1 = fitted["r1"]
    r0 = fitted["r0"]
    m1 = fitted["m1"]
    m0 = fitted["m0"]

    n = len(data)

    # ------------------------------------------------------------
    # Naive complete-case difference in means
    # Descriptive benchmark only
    # ------------------------------------------------------------

    treated_cc = (
        (W == 1) &
        (S == 1) &
        np.isfinite(Y)
    )

    control_cc = (
        (W == 0) &
        (S == 1) &
        np.isfinite(Y)
    )

    naive_cc = (
        np.mean(Y[treated_cc]) -
        np.mean(Y[control_cc])
    )

    # ------------------------------------------------------------
    # OR: average predicted treatment effect
    # ------------------------------------------------------------

    or_estimate = np.mean(m1 - m0)

    # ------------------------------------------------------------
    # Response-adjusted IPW
    # ------------------------------------------------------------

    wt1 = (
        W == 1
    ) * S / (P_RCT * r1)

    wt0 = (
        W == 0
    ) * S / ((1 - P_RCT) * r0)

    ipw1 = np.sum(wt1 * np.nan_to_num(Y)) / np.sum(wt1)
    ipw0 = np.sum(wt0 * np.nan_to_num(Y)) / np.sum(wt0)

    ipw_estimate = ipw1 - ipw0

    # ------------------------------------------------------------
    # DR: response-adjusted randomized-experiment estimator
    # ------------------------------------------------------------

    residual1 = (
        W * S / (P_RCT * r1)
    ) * np.nan_to_num(Y - m1)

    residual0 = (
        (1 - W) * S / ((1 - P_RCT) * r0)
    ) * np.nan_to_num(Y - m0)

    dr_estimate = (
        np.mean(m1 - m0) +
        np.mean(residual1 - residual0)
    )

    return {
        "Naive_Complete_Case": naive_cc,
        "OR": or_estimate,
        "IPW": ipw_estimate,
        "DR": dr_estimate
    }


def run_experimental_analysis(
    outcome_spec="LINEAR"
):

    print("\n" + "=" * 70)
    print("EXPERIMENTAL NSW ANALYSIS")
    print(f"Outcome specification: {outcome_spec}")
    print("=" * 70)

    data = source.copy().reset_index(drop=True)

    fitted_original = fit_rct_models(
        data,
        outcome_spec
    )

    point = calculate_rct_estimators(
        data,
        fitted_original
    )

    bootstrap_rows = []

    successful = 0

    for b in range(B_BOOT):

        boot = data.sample(
            n=len(data),
            replace=True,
            random_state=int(rng.integers(1, 2**31 - 1))
        ).reset_index(drop=True)

        try:

            fitted_boot = fit_rct_models(
                boot,
                outcome_spec
            )

            estimates = calculate_rct_estimators(
                boot,
                fitted_boot
            )

            for estimator, estimate in estimates.items():

                bootstrap_rows.append({
                    "Outcome_Spec": outcome_spec,
                    "Bootstrap": b + 1,
                    "Estimator": estimator,
                    "Estimate": estimate
                })

            successful += 1

        except Exception as error:

            print(
                f"RCT bootstrap {b + 1} failed: {error}"
            )

    bootstrap_df = pd.DataFrame(bootstrap_rows)

    summary_rows = []

    for estimator, estimate in point.items():

        values = bootstrap_df[
            bootstrap_df["Estimator"] == estimator
        ]["Estimate"].dropna()

        summary_rows.append({
            "Population": "NSW Experimental",
            "Outcome_Spec": outcome_spec,
            "Estimator": estimator,
            "Estimate": estimate,
            "Bootstrap_SE": values.std(ddof=1),
            "CI_Lower": values.quantile(0.025),
            "CI_Upper": values.quantile(0.975),
            "Successful_Bootstrap": len(values)
        })

    summary = pd.DataFrame(summary_rows)

    bootstrap_df.to_csv(
        os.path.join(
            BOOT_DIR,
            f"Bootstrap_NSW_{outcome_spec}.csv"
        ),
        index=False
    )

    print(
        f"Successful NSW bootstrap replications: "
        f"{successful}/{B_BOOT}"
    )

    return summary, bootstrap_df


# ================================================================
# 14. RUN ALL ANALYSES
# ================================================================

all_transport_results = []
all_diagnostics = []
all_bootstrap_results = []

original_fits = {}

for outcome_spec in OUTCOME_SPECS:

    # Experimental NSW benchmark
    rct_summary, rct_boot = run_experimental_analysis(
        outcome_spec
    )

    rct_summary.to_csv(
        os.path.join(
            TABLE_DIR,
            f"NSW_Experimental_{outcome_spec}.csv"
        ),
        index=False
    )

    # PSID-1 and PSID-2 transport analyses
    for target_name, target_data in TARGETS.items():

        (
            summary,
            diagnostics,
            boot,
            analysis_data,
            fitted
        ) = run_transport_analysis(
            target_name,
            target_data,
            outcome_spec
        )

        all_transport_results.append(summary)
        all_diagnostics.append(diagnostics)
        all_bootstrap_results.append(boot)

        original_fits[
            (target_name, outcome_spec)
        ] = {
            "data": analysis_data,
            "fitted": fitted
        }


# ================================================================
# 15. COMBINE AND SAVE RESULTS
# ================================================================

transport_results = pd.concat(
    all_transport_results,
    ignore_index=True
)

diagnostics_results = pd.concat(
    all_diagnostics,
    ignore_index=True
)

bootstrap_results = pd.concat(
    all_bootstrap_results,
    ignore_index=True
)

transport_results.to_csv(
    os.path.join(
        TABLE_DIR,
        "Table_3_Transport_Estimates.csv"
    ),
    index=False
)

diagnostics_results.to_csv(
    os.path.join(
        TABLE_DIR,
        "Table_4_Overlap_ESS_Diagnostics.csv"
    ),
    index=False
)

bootstrap_results.to_csv(
    os.path.join(
        BOOT_DIR,
        "All_Transport_Bootstrap_Results.csv"
    ),
    index=False
)


# ================================================================
# 16. COVARIATE BALANCE DIAGNOSTICS
# ================================================================

def weighted_mean(x, w):

    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    valid = (
        np.isfinite(x) &
        np.isfinite(w) &
        (w >= 0)
    )

    if valid.sum() == 0 or w[valid].sum() == 0:
        return np.nan

    return np.sum(x[valid] * w[valid]) / np.sum(w[valid])


def calculate_balance(
    data,
    fitted,
    target_name,
    outcome_spec
):

    Z = data["Z"].to_numpy()
    W = data["W"].to_numpy()
    S = data["S"].to_numpy()

    e = fitted["e"]

    p1 = fitted["p1"]
    p0 = fitted["p0"]

    r1 = fitted["r1"]
    r0 = fitted["r0"]

    tau = fitted["tau"]

    target_mask = Z == 0

    rows = []

    for a in A_VALUES:

        phi = overlap_function(tau, a)

        for g in [1, 0]:

            pg = p1 if g == 1 else p0
            rg = r1 if g == 1 else r0

            omega = ((1 - e) / e) / (pg * rg)

            source_arm = (
                (Z == 1) &
                (W == g) &
                (S == 1)
            )

            source_weights = (
                phi[source_arm] *
                omega[source_arm]
            )

            target_weights = phi[target_mask]

            for variable in COVARIATES:

                source_x = data.loc[
                    source_arm, variable
                ].to_numpy(dtype=float)

                target_x = data.loc[
                    target_mask, variable
                ].to_numpy(dtype=float)

                source_mean = weighted_mean(
                    source_x,
                    source_weights
                )

                target_mean = weighted_mean(
                    target_x,
                    target_weights
                )

                pooled_sd = np.sqrt(
                    (
                        np.nanvar(source_x, ddof=1) +
                        np.nanvar(target_x, ddof=1)
                    ) / 2
                )

                if pooled_sd > 0:
                    smd = (
                        source_mean - target_mean
                    ) / pooled_sd
                else:
                    smd = np.nan

                rows.append({
                    "Target": target_name,
                    "Outcome_Spec": outcome_spec,
                    "a": a,
                    "Treatment_Arm": g,
                    "Covariate": variable,
                    "Source_Weighted_Mean": source_mean,
                    "Target_Weighted_Mean": target_mean,
                    "SMD": smd,
                    "Absolute_SMD": abs(smd)
                })

    return pd.DataFrame(rows)


all_balance = []

for (target_name, outcome_spec), objects in original_fits.items():

    balance = calculate_balance(
        objects["data"],
        objects["fitted"],
        target_name,
        outcome_spec
    )

    all_balance.append(balance)

balance_results = pd.concat(
    all_balance,
    ignore_index=True
)

balance_results.to_csv(
    os.path.join(
        TABLE_DIR,
        "Table_5_Covariate_Balance.csv"
    ),
    index=False
)


# ================================================================
# 17. FIGURE 1: TRANSPORTED EFFECT BY TUNING PARAMETER
# ================================================================

plot_data = transport_results[
    transport_results["Outcome_Spec"] == "LINEAR"
].copy()

for target_name in ["PSID-1", "PSID-2"]:

    fig, ax = plt.subplots(figsize=(9, 6))

    part = plot_data[
        (plot_data["Target"] == target_name) &
        (plot_data["Track"] == "ESTIMATED_OVERLAP")
    ]

    for estimator in ["OR", "IPW", "DR"]:

        temp = part[
            part["Estimator"] == estimator
        ].sort_values("a")

        ax.plot(
            temp["a"],
            temp["Estimate"],
            marker="o",
            linewidth=2,
            label=estimator
        )

        ax.fill_between(
            temp["a"].to_numpy(dtype=float),
            temp["CI_Lower"].to_numpy(dtype=float),
            temp["CI_Upper"].to_numpy(dtype=float),
            alpha=0.12
        )

    ax.set_title(
        f"{target_name}: Adaptive Overlap-Weighted "
        "Transported Treatment Effects"
    )

    ax.set_xlabel("Adaptive overlap tuning parameter (a)")
    ax.set_ylabel("Estimated transported treatment effect")
    ax.legend()

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            FIG_DIR,
            f"Figure_1_{target_name.replace('-', '')}_"
            "Transported_Effects.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ================================================================
# 18. FIGURE 2: EFFECTIVE SAMPLE SIZE
# ================================================================

ess_plot = diagnostics_results[
    diagnostics_results["Outcome_Spec"] == "LINEAR"
].copy()

ess_plot = ess_plot[
    ess_plot["Track"] == "ESTIMATED_OVERLAP"
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5)
)

for ax, target_name in zip(
    axes,
    ["PSID-1", "PSID-2"]
):

    part = ess_plot[
        ess_plot["Target"] == target_name
    ]

    ax.plot(
        part["a"],
        part["ESS1"],
        marker="o",
        label="Treated ESS"
    )

    ax.plot(
        part["a"],
        part["ESS0"],
        marker="s",
        label="Control ESS"
    )

    ax.set_title(f"{target_name}: Effective Sample Size")
    ax.set_xlabel("Adaptive overlap tuning parameter (a)")
    ax.set_ylabel("Effective sample size")
    ax.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_2_ESS.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ================================================================
# 19. FIGURE 3: MEAN OVERLAP FUNCTION
# ================================================================

fig, ax = plt.subplots(figsize=(8, 6))

for target_name in ["PSID-1", "PSID-2"]:

    part = ess_plot[
        ess_plot["Target"] == target_name
    ].sort_values("a")

    ax.plot(
        part["a"],
        part["Mean_Phi"],
        marker="o",
        linewidth=2,
        label=target_name
    )

ax.set_title("Mean Adaptive Overlap Function")
ax.set_xlabel("Adaptive overlap tuning parameter (a)")
ax.set_ylabel("Mean overlap function")
ax.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_3_Mean_Overlap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ================================================================
# 20. FIGURE 4: COVARIATE BALANCE / LOVE PLOT
# ================================================================

love_data = balance_results[
    (balance_results["Outcome_Spec"] == "LINEAR") &
    (balance_results["a"].isin([0.0, 0.05, 0.10]))
].copy()

for target_name in ["PSID-1", "PSID-2"]:

    for arm in [1, 0]:

        part = love_data[
            (love_data["Target"] == target_name) &
            (love_data["Treatment_Arm"] == arm)
        ]

        fig, ax = plt.subplots(figsize=(9, 7))

        for a in [0.0, 0.05, 0.10]:

            temp = part[
                part["a"] == a
            ]

            ax.scatter(
                temp["Absolute_SMD"],
                temp["Covariate"],
                label=f"a = {a}",
                s=55
            )

        ax.axvline(
            0.1,
            linestyle="--",
            linewidth=1.5,
            label="SMD = 0.10"
        )

        ax.set_title(
            f"{target_name}: Covariate Balance, "
            f"Treatment Arm {arm}"
        )

        ax.set_xlabel("Absolute standardized mean difference")
        ax.set_ylabel("Covariate")
        ax.legend()

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                FIG_DIR,
                f"Figure_4_Love_{target_name.replace('-', '')}_"
                f"Arm_{arm}.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        plt.close()


# ================================================================
# 21. FIGURE 5: ADAPTIVE WEIGHT DISTRIBUTION
# ================================================================

for target_name in ["PSID-1", "PSID-2"]:

    objects = original_fits[
        (target_name, "LINEAR")
    ]

    data = objects["data"]
    fitted = objects["fitted"]

    Z = data["Z"].to_numpy()
    W = data["W"].to_numpy()
    S = data["S"].to_numpy()

    e = fitted["e"]

    for a in [0.0, 0.05, 0.10]:

        phi = overlap_function(
            fitted["tau"], a
        )

        fig, ax = plt.subplots(figsize=(9, 6))

        for g in [1, 0]:

            pg = (
                fitted["p1"] if g == 1
                else fitted["p0"]
            )

            rg = (
                fitted["r1"] if g == 1
                else fitted["r0"]
            )

            omega = ((1 - e) / e) / (pg * rg)

            mask = (
                (Z == 1) &
                (W == g) &
                (S == 1)
            )

            adaptive_weights = (
                phi[mask] * omega[mask]
            )

            ax.hist(
                np.log10(
                    np.maximum(adaptive_weights, EPS)
                ),
                bins=30,
                alpha=0.55,
                label=f"Treatment arm {g}"
            )

        ax.set_title(
            f"{target_name}: Adaptive Weight Distribution "
            f"(a = {a})"
        )

        ax.set_xlabel("Log10 adaptive weight")
        ax.set_ylabel("Frequency")
        ax.legend()

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                FIG_DIR,
                f"Figure_5_Weight_Distribution_"
                f"{target_name.replace('-', '')}_a_{a}.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        plt.close()


# ================================================================
# 22. FIGURE 6: EXPERIMENTAL NSW ESTIMATORS
# ================================================================

rct_files = [
    os.path.join(
        TABLE_DIR,
        f"NSW_Experimental_{spec}.csv"
    )
    for spec in OUTCOME_SPECS
]

rct_plot = pd.concat(
    [pd.read_csv(f) for f in rct_files],
    ignore_index=True
)

rct_plot = rct_plot[
    rct_plot["Outcome_Spec"] == "LINEAR"
]

fig, ax = plt.subplots(figsize=(9, 6))

for _, row in rct_plot.iterrows():

    ax.errorbar(
        row["Estimate"],
        row["Estimator"],
        xerr=[[
            row["Estimate"] - row["CI_Lower"]
        ], [
            row["CI_Upper"] - row["Estimate"]
        ]],
        fmt="o",
        capsize=4
    )

ax.set_title(
    "Experimental NSW Treatment Effect Estimates"
)

ax.set_xlabel("Estimated NSW average treatment effect")
ax.set_ylabel("Estimator")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_6_NSW_Experimental_Benchmark.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ================================================================
# 23. FIXED-SCORE VS ESTIMATED-OVERLAP COMPARISON
# ================================================================

fixed_comparison = transport_results.pivot_table(
    index=[
        "Target",
        "Outcome_Spec",
        "a",
        "Estimator"
    ],
    columns="Track",
    values="Estimate"
).reset_index()

fixed_comparison["Difference"] = (
    fixed_comparison["FIXED_SCORE"] -
    fixed_comparison["ESTIMATED_OVERLAP"]
)

fixed_comparison.to_csv(
    os.path.join(
        TABLE_DIR,
        "Table_6_Fixed_vs_Estimated_Overlap.csv"
    ),
    index=False
)


# ================================================================
# 24. LINEAR VS QUADRATIC OUTCOME MODEL SENSITIVITY
# ================================================================

sensitivity = transport_results.pivot_table(
    index=[
        "Target",
        "a",
        "Track",
        "Estimator"
    ],
    columns="Outcome_Spec",
    values="Estimate"
).reset_index()

if "LINEAR" in sensitivity.columns and "QUADRATIC" in sensitivity.columns:

    sensitivity["Quadratic_minus_Linear"] = (
        sensitivity["QUADRATIC"] -
        sensitivity["LINEAR"]
    )

sensitivity.to_csv(
    os.path.join(
        TABLE_DIR,
        "Table_7_Outcome_Model_Sensitivity.csv"
    ),
    index=False
)


# ================================================================
# 25. EXCEL WORKBOOK
# ================================================================

excel_path = os.path.join(
    OUTPUT_DIR,
    "Complete_Empirical_Analysis_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    sample_flow.to_excel(
        writer,
        sheet_name="Sample_Flow",
        index=False
    )

    descriptive_table.to_excel(
        writer,
        sheet_name="Descriptive_Statistics",
        index=False
    )

    transport_results.to_excel(
        writer,
        sheet_name="Transport_Estimates",
        index=False
    )

    diagnostics_results.to_excel(
        writer,
        sheet_name="Overlap_ESS",
        index=False
    )

    balance_results.to_excel(
        writer,
        sheet_name="Covariate_Balance",
        index=False
    )

    fixed_comparison.to_excel(
        writer,
        sheet_name="Fixed_vs_Estimated",
        index=False
    )

    sensitivity.to_excel(
        writer,
        sheet_name="Outcome_Sensitivity",
        index=False
    )

    rct_plot.to_excel(
        writer,
        sheet_name="NSW_Experimental",
        index=False
    )


# ================================================================
# 26. SAVE ANALYSIS SETTINGS AND SAMPLE INFORMATION
# ================================================================

settings = pd.DataFrame([
    ["Dataset", os.path.basename(DATA_PATH)],
    ["Source population", "NSW: sample_lalonde == 1"],
    ["Target population 1", "PSID-1: sample_lalonde == 0 and psid1 == 1"],
    ["Target population 2", "PSID-2: sample_lalonde == 0 and psid2 == 1"],
    ["Outcome", OUTCOME],
    ["Treatment", TREATMENT],
    ["Treatment probability", P_RCT],
    ["Bootstrap replications", B_BOOT],
    ["Probability clipping", EPS],
    ["Adaptive overlap values", str(A_VALUES)],
    ["Outcome specifications", str(OUTCOME_SPECS)],
    ["Random seed", SEED],
    ["Source sample size", len(source)],
    ["Source treated", int((source["W"] == 1).sum())],
    ["Source controls", int((source["W"] == 0).sum())],
    ["Source missing outcomes", int(source[OUTCOME].isna().sum())],
    ["PSID-1 sample size", len(target_psid1)],
    ["PSID-2 sample size", len(target_psid2)]
], columns=["Setting", "Value"])

settings.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Analysis_Settings.csv"
    ),
    index=False
)


# ================================================================
# 27. FINAL COMPLETION MESSAGE
# ================================================================

print("\n" + "=" * 75)
print("EMPIRICAL ANALYSIS COMPLETED")
print("=" * 75)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nMain result files:")
print("1. Complete_Empirical_Analysis_Results.xlsx")
print("2. Table_3_Transport_Estimates.csv")
print("3. Table_4_Overlap_ESS_Diagnostics.csv")
print("4. Table_5_Covariate_Balance.csv")
print("5. Table_6_Fixed_vs_Estimated_Overlap.csv")
print("6. Table_7_Outcome_Model_Sensitivity.csv")

print("\nFigures are saved in:")
print(FIG_DIR)

print("\nBootstrap estimates are saved in:")
print(BOOT_DIR)

print("\nPlease inspect bootstrap success counts and diagnostics")
print("before using the estimates in a journal manuscript.")

print("\nALL TASKS FINISHED.")


EXPERIMENTAL NSW ANALYSIS
Outcome specification: LINEAR
Successful NSW bootstrap replications: 1000/1000

TRANSPORT ANALYSIS: PSID-1
Outcome specification: LINEAR
Successful bootstrap replications: 1000/1000

TRANSPORT ANALYSIS: PSID-2
Outcome specification: LINEAR
Successful bootstrap replications: 1000/1000

EXPERIMENTAL NSW ANALYSIS
Outcome specification: QUADRATIC
Successful NSW bootstrap replications: 1000/1000

TRANSPORT ANALYSIS: PSID-1
Outcome specification: QUADRATIC
Successful bootstrap replications: 1000/1000

TRANSPORT ANALYSIS: PSID-2
Outcome specification: QUADRATIC
Successful bootstrap replications: 1000/1000

EMPIRICAL ANALYSIS COMPLETED

Output directory:
D:\empirical analysis

Main result files:
1. Complete_Empirical_Analysis_Results.xlsx
2. Table_3_Transport_Estimates.csv
3. Table_4_Overlap_ESS_Diagnostics.csv
4. Table_5_Covariate_Balance.csv
5. Table_6_Fixed_vs_Estimated_Overlap.csv
6. Table_7_Outcome_Model_Sensitivity.csv

Figures are saved in:
D:\empirical analys